# Meditation and mind wandering in the EEG

**Question:** Can you see in the EEG whether a meditator is absorbed in the meditation or whether the mind has wandered off?

**Data:** EEG from 24 meditators of the Himalayan Yoga tradition, recorded in Rishikesh, India: 12 experts (at least 2 hours of meditation a day for a year or more) and 12 non-experts (Brandmeyer & Delorme, 2018, *Experimental Brain Research*; open data: OpenNeuro ds001787). They meditated with closed eyes for 45 to 90 minutes. Every 30 to 90 seconds, a recorded voice interrupted them with three questions, each answered on a keypad from 0 to 3:
- Q1: how deep was your meditation?
- Q2: how deep was your mind wandering?
- Q3: how tired are you?

**The finding we try to reproduce:** in the 10 s before a question, the experts had more theta power (4–7 Hz) over the front of the head and more alpha power (9–11 Hz) over the somatosensory cortex when they reported meditation than when they reported mind wandering. The non-experts showed no significant difference.

**Tool:** the power spectrum $P(f) = |X(f)|^2$, where $X(f)$ is the Fourier transform of the EEG signal $x(t)$: how much power the signal has at each frequency. We estimate it with Welch's method.

Imports and plot style, the same as in projects 01 and 02. MNE-Python reads the EEG files and knows where each electrode sits on the head.

In [ ]:
from pathlib import Path
import re
import urllib.request
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import mne
from scipy import stats

mne.set_log_level("error")
plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
BLUE = "#2a78d6"    # meditation
ORANGE = "#eb6834"  # mind wandering
GRAY = "#52514e"    # reference lines, labels

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data") / "ds001787"

Download the dataset from OpenNeuro (40 recording sessions, about 6 GB), but only the files that are not on disk yet. We skip the sound files of the spoken questions. The data folder is not part of the repository.

In [ ]:
BUCKET = "https://s3.amazonaws.com/openneuro.org"

def list_dataset_files():
    """All files of ds001787 on OpenNeuro with their sizes in bytes, without sounds and metadata."""
    xml = urllib.request.urlopen(f"{BUCKET}?list-type=2&prefix=ds001787/").read().decode()
    files = re.findall(r"<Key>ds001787/(.*?)</Key>.*?<Size>(\d+)</Size>", xml)
    return [(rel, int(size)) for rel, size in files
            if not rel.startswith(("stimuli/", ".datalad/")) and not rel.endswith(".gitattributes")]

files = list_dataset_files()
for rel, size in files:
    path = DATA_DIR / rel
    if path.exists() and path.stat().st_size == size:
        continue
    path.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", rel)
    urllib.request.urlretrieve(f"{BUCKET}/ds001787/{rel}", path)
print(f"{len(files)} files, {sum(s for _, s in files) / 1e9:.1f} GB in {DATA_DIR}")

Who took part, and how many recording sessions each person has (one to three).

In [ ]:
rows = [line.split("\t") for line in (DATA_DIR / "participants.tsv").read_text().splitlines()[1:]]
group = {r[0]: r[3] for r in rows}  # "sub-001" -> "expert" or "novice"
sessions = {sub: sorted(p.name for p in (DATA_DIR / sub).glob("ses-*")) for sub in group}

experts = [s for s in group if group[s] == "expert"]
novices = [s for s in group if group[s] == "novice"]
print(f"{len(experts)} experts, {len(novices)} non-experts, "
      f"{sum(len(v) for v in sessions.values())} sessions")

## The answers

The EEG file marks the moment each question starts (marker 128). Its answer markers, however, do not say which question they answer, and answers of 0 are missing. The log files written by the experiment program have all three answers per question, each with its time. We read the answers from there.

In [ ]:
LOG_ZIP = DATA_DIR / "code" / "MW_Current_TextFileBIDS.zip"

def parse_log(text):
    """Questions in one log file: time (s, log clock) and the answers q1, q2, q3 (0-3)."""
    probes = []
    for line in text.splitlines():
        m = re.match(r"MW question asked at time\s+([\d.]+)", line)
        if m:
            probes.append({"t": float(m.group(1))})
            continue
        m = re.match(r"Key ([0-3]) ?\(code \d\) pressed at time\s+[\d.]+ seconds, status ([123])", line)
        if m and probes:
            probes[-1][f"q{m.group(2)}"] = int(m.group(1))   # status 1, 2, 3 = question 1, 2, 3
    return probes

with zipfile.ZipFile(LOG_ZIP) as z:
    example = parse_log(z.read("MW_Current_TextFileBIDS/sub01_info.txt").decode())
for p in example[:4]:
    print(p)

The log clock and the EEG clock started at different moments. We find the shift by matching the question times: the log file whose question times line up with the EEG markers after one constant shift belongs to this recording. Each question then gets the exact time of its EEG marker. We search all 41 log files, not only the one with the person's number, because one log file has the wrong name: the questions recorded in the EEG of sub-013 match the log file `sub14_info.txt` (25 of 25), while `sub13_info.txt` matches no recording.

In [ ]:
def eeg_file(sub, ses, suffix):
    return DATA_DIR / sub / ses / "eeg" / f"{sub}_{ses}_task-meditation_{suffix}"

with zipfile.ZipFile(LOG_ZIP) as z:
    LOGS = {n.split("/")[1]: parse_log(z.read(n).decode(errors="replace"))
            for n in z.namelist() if re.fullmatch(r"MW_Current_TextFileBIDS/sub\d+_.*info\.txt", n)}

def find_answers(sub, ses, tol=0.3):
    """Questions of one recording with their answers, timed on the EEG clock (s),
    plus the name of the matching log file and the clock shift (s)."""
    events = np.genfromtxt(eeg_file(sub, ses, "events.tsv"), delimiter="\t",
                           names=True, dtype=None, encoding=None)
    onsets = events["onset"][events["value"] == 128]          # question onsets in the EEG
    best_hits, best = 0, None
    for name, probes in LOGS.items():
        t_log = np.array([p["t"] for p in probes])
        for shift in (onsets[:, None] - t_log[None, :]).ravel():   # try every pairing
            dist = np.abs(onsets[:, None] - (t_log[None, :] + shift))
            hits = np.sum(dist.min(axis=0) < tol)
            if hits > best_hits:
                best_hits, best = hits, (name, probes, t_log, shift)
    name, probes, t_log, shift = best
    matched = []
    for p, t in zip(probes, t_log):
        i = np.argmin(np.abs(onsets - (t + shift)))
        if abs(onsets[i] - (t + shift)) < tol:
            matched.append({**p, "t": onsets[i]})
    return matched, name, shift

answers, log_name, shift = find_answers("sub-001", "ses-01")
print(f"sub-001 ses-01: {len(answers)} questions matched in {log_name}, "
      f"log clock shifted by {shift:.1f} s")

**Meditation or mind wandering?** As in the paper, a question counts as *meditation* when the meditation depth (Q1) is higher than the mind-wandering depth (Q2), and as *mind wandering* when Q2 is higher. Equal ratings are left out.

In [ ]:
def label(q1, q2):
    """'med', 'mw', or None when both ratings are equal."""
    return "med" if q1>q2 else "mw" if q2>q1 else None

labels = [label(p["q1"], p["q2"]) for p in answers if "q1" in p and "q2" in p]
print(f"meditation: {labels.count('med')}, mind wandering: {labels.count('mw')}, "
      f"equal: {labels.count(None)}")

## The EEG

Load one expert session. The file has 64 EEG channels with BioSemi names (A1 ... B32); we give them their standard names and positions on the head. Then we filter as in the paper: a high-pass filter at 2 Hz removes slow drifts, and the average reference subtracts the mean of all 64 channels from each one.

A broken electrode (bad skin contact) shows up as a channel whose typical swing is several times larger than that of the others. Before averaging, we rebuild such a channel from its neighbours (spherical-spline interpolation); otherwise its noise would leak into every channel through the average reference. In the 40 sessions this rule catches two channels: FT7 in sub-003 (second session) and P2 in sub-012.

In [ ]:
CH_NAMES = mne.channels.make_standard_montage("biosemi64").ch_names

def load_eeg(sub, ses):
    """64 EEG channels in volts, standard names, 2 Hz high-pass, broken channels interpolated, average reference."""
    raw = mne.io.read_raw_bdf(eeg_file(sub, ses, "eeg.bdf"), preload=True)
    raw.pick(raw.ch_names[:64])                         # A1 ... B32 are the EEG channels
    raw.rename_channels(dict(zip(raw.ch_names, CH_NAMES)))
    raw.set_montage("biosemi64")
    raw.filter(l_freq=2, h_freq=None)
    n = int(10 * raw.info["sfreq"])                     # cut the recording into 10-s pieces
    pieces = raw.get_data()[:, : raw.n_times // n * n].reshape(64, -1, n)
    size = np.median(np.ptp(pieces, axis=2), axis=1)    # typical peak-to-peak of each channel
    raw.info["bads"] = [ch for ch, s in zip(CH_NAMES, size) if s > 3 * np.median(size)]
    raw.interpolate_bads()                              # rebuild broken channels from their neighbours
    raw.set_eeg_reference("average")
    return raw

raw = load_eeg("sub-001", "ses-01")
sf = raw.info["sfreq"]
print(f"{len(raw.ch_names)} channels, {sf:.0f} Hz, {raw.n_times / sf / 60:.0f} min")

**First look:** the raw EEG in the 10 s before one question, at four sites along the midline from the forehead (Fz) to the back of the head (Oz).

In [ ]:
p = next(p for p in answers if label(p["q1"], p["q2"]) == "med")   # first meditation question
start, stop = int((p["t"] - 10) * sf), int(p["t"] * sf)
sites = ["Fz", "Cz", "Pz", "Oz"]
x = raw.get_data(picks=sites, start=start, stop=stop) * 1e6        # volts -> µV
t = np.arange(x.shape[1]) / sf - 10

fig, ax = plt.subplots(figsize=(9, 4.5))
gap = 60  # µV between traces
for k, (site, trace) in enumerate(zip(sites, x)):
    ax.plot(t, trace - k * gap, color=BLUE, lw=0.8)
    ax.text(-10.2, -k * gap, site, ha="right", va="center", color=GRAY)
ax.plot([0.25, 0.25], [-4 * gap, -4 * gap + 50], color=GRAY, lw=2, clip_on=False)  # 50 µV
ax.text(0.35, -4 * gap + 25, "50 µV", va="center", color=GRAY, fontsize=9, clip_on=False)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_xlim(-10, 0)
ax.set_xlabel("time before the question (s)")
ax.set_title(f"sub-001, 10 s before a question (meditation {p['q1']}, mind wandering {p['q2']})")
fig.tight_layout()
fig.savefig(FIG_DIR / "eeg_before_question.png", dpi=150)
plt.show()

The raw EEG is a mix of waves of a few tens of µV at all four sites. By eye, you cannot read the mental state from it, so we split it into its frequencies.

## Power spectrum

For every question we take the 10 s before it (from −10.05 to −0.05 s, as in the paper) and compute the power spectrum with Welch's method: cut the 10 s into 2-s pieces that overlap by half, compute $|X(f)|^2$ for each piece and average. Averaging the pieces makes the spectrum less noisy, just as averaging trials made the PSTH less noisy in project 02. Pieces of 2 s give a frequency resolution of 0.5 Hz.

In [ ]:
THETA = (4, 7)   # Hz
ALPHA = (9, 11)  # Hz

def question_spectra(raw, answers):
    """For the 10 s before each question: spectra (questions, channels, frequencies) in µV²/Hz,
    peak-to-peak size of each channel in µV (to spot artifacts later), labels, and frequencies."""
    sf = raw.info["sfreq"]
    spectra, sizes, labs = [], [], []
    for p in answers:
        start = int(round((p["t"] - 10.05) * sf))
        stop = int(round((p["t"] - 0.05) * sf))
        if start < 0 or "q1" not in p or "q2" not in p:
            continue
        x = raw.get_data(start=start, stop=stop) * 1e6          # volts -> µV
        psd, freqs = mne.time_frequency.psd_array_welch(
            x, sf, fmin=1, fmax=40, n_fft=int(2 * sf), n_overlap=int(sf))
        spectra.append(psd)
        sizes.append(np.ptp(x, axis=1))
        labs.append(label(p["q1"], p["q2"]) or "equal")         # None -> "equal"
    return np.array(spectra), np.array(sizes), np.array(labs), freqs

spectra, sizes, labs, freqs = question_spectra(raw, answers)
print(f"{spectra.shape[0]} questions x {spectra.shape[1]} channels x {spectra.shape[2]} frequencies "
      f"({freqs[0]:.1f} to {freqs[-1]:.1f} Hz in steps of {freqs[1] - freqs[0]:.1f} Hz)")

Left: the average spectrum at Oz over all questions of this session, with the two bands of the paper shaded. Right: the same at Fz, split into meditation and mind wandering.

In [ ]:
med, mw = labs == "med", labs == "mw"
oz, fz = CH_NAMES.index("Oz"), CH_NAMES.index("Fz")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax in axes:
    for (lo, hi), name in [(THETA, "theta"), (ALPHA, "alpha")]:
        ax.axvspan(lo, hi, color=GRAY, alpha=0.12, lw=0)
        ax.text((lo + hi) / 2, 0.02, name, transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", color=GRAY, fontsize=9)
    ax.set_yscale("log")
    ax.set_xlim(2, 40)   # below 2 Hz the high-pass filter has removed the signal
    ax.set_xlabel("frequency (Hz)")

ax = axes[0]
mean_oz = spectra[:, oz].mean(axis=0)
ax.plot(freqs, mean_oz, color=GRAY, lw=2)
f_peak = freqs[(freqs > 5)][np.argmax(mean_oz[freqs > 5])]
ax.set_ylabel("power (µV²/Hz)")
ax.set_title(f"Oz, all {len(spectra)} questions: peak at {f_peak:.1f} Hz", fontsize=11)

ax = axes[1]
ax.plot(freqs, spectra[med, fz].mean(axis=0), color=BLUE, lw=2, label=f"meditation ({med.sum()})")
ax.plot(freqs, spectra[mw, fz].mean(axis=0), color=ORANGE, lw=2, label=f"mind wandering ({mw.sum()})")
ax.legend(frameon=False)
ax.set_title("Fz, one expert session", fontsize=11)

fig.tight_layout()
fig.savefig(FIG_DIR / "spectrum_one_expert.png", dpi=150)
plt.show()

At the back of the head (Oz), the spectrum has a sharp alpha peak at 10.5 Hz. At the front (Fz), there is a second peak at 5.5 Hz, inside the theta band. In this one session, theta at Fz is a little higher before meditation reports (2.8 vs 2.3 µV²/Hz), but with only 8 questions on each side this says nothing yet; that needs all 12 experts.

## All 24 meditators

Now the same for every question in all 40 sessions. Loading and filtering all of them takes a few minutes, so the result is saved once to `data/question_spectra.npz` and read from there afterwards.

In [ ]:
CACHE = Path("data") / "question_spectra.npz"

if not CACHE.exists():
    parts = {"spectra": [], "sizes": [], "labels": [], "person": []}
    for sub in group:
        for ses in sessions[sub]:
            answers_ses, _, _ = find_answers(sub, ses)
            s, z, l, freqs = question_spectra(load_eeg(sub, ses), answers_ses)
            parts["spectra"].append(s.astype(np.float32))
            parts["sizes"].append(z.astype(np.float32))
            parts["labels"].append(l)
            parts["person"].append(np.full(len(l), sub))
            print(f"{sub} {ses}: {len(l)} questions")
    np.savez_compressed(CACHE, freqs=freqs, **{k: np.concatenate(v) for k, v in parts.items()})

saved = np.load(CACHE)
S, sizes_all, labels_all, person = saved["spectra"], saved["sizes"], saved["labels"], saved["person"]
freqs = saved["freqs"]
print(f"{len(S)} questions from {len(np.unique(person))} people")

**Behaviour, per person.** The paper reports the share of meditation answers as a mean over people (experts 75.4 %, non-experts 42.7 %). Two people never gave one of the two answers, so they cannot be compared with themselves; the paper also leaves out two people for this reason.

In [ ]:
def share_meditation(sub):
    """Share of meditation among this person's meditation and mind-wandering answers (%)."""
    mine = labels_all[(person == sub) & (labels_all != "equal")]
    return 100 * np.mean(mine == "med")

for name, people in [("experts", experts), ("non-experts", novices)]:
    shares = [share_meditation(s) for s in people]
    print(f"{name}: {np.mean(shares):.1f} % meditation (mean over {len(people)} people)")
for sub in group:
    n_med = np.sum((person == sub) & (labels_all == "med"))
    n_mw = np.sum((person == sub) & (labels_all == "mw"))
    if n_med == 0 or n_mw == 0:
        print(f"{sub} ({group[sub]}): {n_med} meditation, {n_mw} mind wandering -> left out")

The experts' share matches the paper closely (74.6 % vs 75.4 %); the non-experts' share is 4.5 points lower (38.2 % vs 42.7 %), for a reason we do not know. As in the paper, two people are left out of the EEG comparison, both non-experts: one never reported mind wandering, the other never meditation.

**Artifacts.** Some 10-s windows contain large artifacts: eye movements, tense muscles, a moving cable. The paper removed them by hand and with independent component analysis (ICA). We use a simple automatic rule instead: a window is dropped if any channel swings more than 200 µV from its lowest to its highest point.

In [ ]:
MAX_SIZE = 200   # µV peak to peak
keep = sizes_all.max(axis=1) <= MAX_SIZE
print(f"kept {keep.sum()} of {len(keep)} questions ({100 * keep.mean():.0f} %)")
for name, people in [("experts", experts), ("non-experts", novices)]:
    for lab in ["med", "mw"]:
        mine = np.isin(person, people) & (labels_all == lab)
        print(f"{name}, {lab}: {100 * np.mean(~keep[mine]):.0f} % of windows dropped")

The rule drops 139 of 971 windows (14 %), but not evenly. For the experts it drops 9 % of the windows before meditation reports and 12 % before mind-wandering reports; for the non-experts 8 % and 28 %. So the non-experts moved or tensed up much more often before they reported mind wandering. That is a small finding of its own, and a warning: here, artifacts and mental state go together.

**One comparison per person.** EEG power differs a lot between people (skull thickness, skin contact), much more than between two mental states of the same person. So, as in the paper, every person is compared with themself: average the band power over all their meditation windows and over all their mind-wandering windows, convert both to decibels, $10 \log_{10} P$, and take the difference. A difference of +0.5 dB means 12 % more power during meditation ($10^{0.05} = 1.12$).

In [ ]:
def band_power_db(people, band, use):
    """Per person and channel: band power in dB for meditation and for mind wandering,
    using only the windows marked True in `use`. People without a window of each kind are left out."""
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    med, mw, used = [], [], []
    for sub in people:
        is_med = (person == sub) & (labels_all == "med") & use
        is_mw = (person == sub) & (labels_all == "mw") & use
        if is_med.sum() == 0 or is_mw.sum() == 0:
            continue
        med.append(10 * np.log10(S[is_med][:, :, in_band].mean(axis=(0, 2))))
        mw.append(10 * np.log10(S[is_mw][:, :, in_band].mean(axis=(0, 2))))
        used.append(sub)
    return np.array(med), np.array(mw), used

med, mw, used = band_power_db(experts, THETA, keep)
print(f"theta, {len(used)} experts: meditation minus mind wandering, mean over people and channels "
      f"{(med - mw).mean():+.2f} dB")

**Is it more than noise?** With 64 channels, about 3 would pass p < 0.05 by chance alone. The paper therefore used a cluster test (Maris & Oostenveld, 2007): neighbouring channels that all differ in the same direction form a cluster, and the cluster's size is compared with the clusters that appear when meditation and mind wandering are swapped at random within each person. With 12 people there are only $2^{12} = 4096$ ways to swap, so we try them all.

In [ ]:
info = mne.create_info(CH_NAMES, sf, "eeg")
info.set_montage("biosemi64")
adjacency, _ = mne.channels.find_ch_adjacency(info, "eeg")

def cluster_test(diff):
    """Channels inside a cluster with p < 0.05, and the p-value of every cluster."""
    _, clusters, p_values, _ = mne.stats.permutation_cluster_1samp_test(
        diff, adjacency=adjacency, n_permutations=10000, tail=0, seed=0, out_type="mask")
    significant = np.zeros(len(CH_NAMES), dtype=bool)
    for cluster, p in zip(clusters, p_values):
        if p < 0.05:
            significant |= cluster
    return significant, p_values

Head maps in the layout of the paper's Figure 4: band power during meditation, during mind wandering, and the difference. The nose points up. In the difference maps, blue means more power during meditation, orange more power during mind wandering; black dots mark channels in a significant cluster.

In [ ]:
DIFF_CMAP = LinearSegmentedColormap.from_list("mw_to_med", [ORANGE, "#f2f1ee", BLUE])
DIFF_LIMIT = 1.0   # dB, the same for both groups
POWER_CMAP = LinearSegmentedColormap.from_list("power", ["#ebe6f5", "#3f1f7a"])   # one hue, light to dark

def plot_head_maps(people, name, filename):
    fig, axes = plt.subplots(2, 3, figsize=(10, 6.4), layout="constrained")
    for row, (band_name, band) in enumerate([("theta", THETA), ("alpha", ALPHA)]):
        med, mw, used = band_power_db(people, band, keep)
        diff = med - mw
        significant, p_values = cluster_test(diff)
        lo, hi = np.percentile(np.r_[med.mean(axis=0), mw.mean(axis=0)], [2, 98])
        for col, data in enumerate([med.mean(axis=0), mw.mean(axis=0)]):
            im, _ = mne.viz.plot_topomap(data, info, axes=axes[row, col], vlim=(lo, hi),
                                         cmap=POWER_CMAP, contours=0, sensors=False, show=False)
        fig.colorbar(im, ax=axes[row, :2], shrink=0.75, label="power (dB)")
        im, _ = mne.viz.plot_topomap(diff.mean(axis=0), info, axes=axes[row, 2],
                                     vlim=(-DIFF_LIMIT, DIFF_LIMIT), cmap=DIFF_CMAP, contours=0,
                                     sensors=False, mask=significant,
                                     mask_params=dict(marker="o", markersize=4, markerfacecolor="black",
                                                      markeredgecolor="black"), show=False)
        fig.colorbar(im, ax=axes[row, 2], shrink=0.75, label="meditation − mind wandering (dB)")
        axes[row, 0].text(-0.12, 0.5, f"{band_name} {band[0]}–{band[1]} Hz", rotation=90, fontsize=12,
                          ha="center", va="center", transform=axes[row, 0].transAxes)
        axes[row, 2].text(0.5, -0.06, f"smallest cluster p = {min(p_values):.2f}" if len(p_values)
                          else "no cluster", fontsize=9, color=GRAY, ha="center", va="top",
                          transform=axes[row, 2].transAxes)
    for ax, title in zip(axes[0], ["meditation", "mind wandering", "difference"]):
        ax.set_title(title)
    fig.suptitle(f"{name}: {len(used)} people", fontsize=13)
    fig.savefig(FIG_DIR / filename, dpi=150)
    plt.show()

plot_head_maps(experts, "Experts", "head_maps_experts.png")

**Experts:** in both bands the difference is positive (blue) over most of the head: more theta and more alpha before meditation reports than before mind-wandering reports. That is the direction of the paper, with a similar size (its difference maps span about ±0.6 dB), and the alpha difference is largest over the right central region, as in the paper. But no cluster passes the test (smallest cluster p = 0.25 for theta, 0.20 for alpha), whereas the paper found clusters with p < 0.02. Only at the forehead is theta higher before mind wandering; these electrodes sit just above the eyes, so eye movements are one possible source, and the paper removed those with ICA.

In [ ]:
plot_head_maps(novices, "Non-experts", "head_maps_nonexperts.png")

**Non-experts:** theta shows no clear difference (smallest cluster p = 0.30). Alpha, however, is lower before meditation than before mind wandering over most of the head, and this cluster passes the test (p = 0.01). The paper reports no significant difference for non-experts. We do not know why yet; the robustness check below shows that this result also depends on our choices.

**One fair test per band.** Testing all 64 channels costs statistical power. The paper's significant channels form two regions (its Figure 4): mid-frontal for theta and right central ("somatosensory") for alpha. We average our differences over these regions, so each person gives one number per band. Because the regions come from the paper and not from our data, this is one honest test per band and group.

In [ ]:
REGIONS = {
    "theta": (THETA, ["F1", "Fz", "F2", "FC1", "FCz", "FC2", "C1", "Cz", "C2"], "mid-frontal"),
    "alpha": (ALPHA, ["FC4", "FC6", "C4", "C6", "CP4", "CP6"], "right central"),
}

def region_difference(people, band, region, use):
    """One number per person: meditation minus mind wandering (dB), averaged over the region's channels."""
    med, mw, _ = band_power_db(people, band, use)
    idx = [CH_NAMES.index(ch) for ch in region]
    return (med - mw)[:, idx].mean(axis=1)

rng = np.random.default_rng(0)   # sideways scatter of the dots, only for readability
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
for ax, (band_name, (band, region, where)) in zip(axes, REGIONS.items()):
    ticks = []
    for x, (name, people) in enumerate([("experts", experts), ("non-experts", novices)]):
        d = region_difference(people, band, region, keep)
        t, p = stats.ttest_1samp(d, 0)
        print(f"{band_name}, {name}: mean {d.mean():+.2f} dB, {np.sum(d > 0)} of {len(d)} above 0, "
              f"t = {t:.2f}, p = {p:.3f}")
        ax.scatter(x + rng.uniform(-0.13, 0.13, len(d)), d, s=36, zorder=3,
                   color=[BLUE if v > 0 else ORANGE for v in d])
        ax.plot([x - 0.22, x + 0.22], [d.mean()] * 2, color="black", lw=2.5)
        ticks.append(f"{name}\n{np.sum(d > 0)} of {len(d)} above 0, p = {p:.2f}")
    ax.axhline(0, color=GRAY, lw=1, ls="--")
    ax.set_xticks([0, 1], ticks)
    ax.set_xlim(-0.5, 1.5)
    ax.set_title(f"{band_name} {band[0]}–{band[1]} Hz, {where}", fontsize=11)
axes[0].set_ylabel("meditation − mind wandering (dB)")
fig.suptitle("One dot per person; black bar = mean", fontsize=11, color=GRAY)
fig.tight_layout()
fig.savefig(FIG_DIR / "regions_per_person.png", dpi=150)
plt.show()

In the paper's theta region, 9 of 12 experts have more theta before meditation (mean +0.39 dB, p = 0.04); in its alpha region, 8 of 12 (mean +0.59 dB, p = 0.09). The spread between people is large: the experts' alpha differences range from about −0.8 to +2.9 dB. For non-experts, theta shows nothing (p = 0.19), and alpha tends the other way (2 of 10 above zero, p = 0.06).

**How robust is this?** An analysis has many small choices, and a p-value near 0.05 can flip with any of them. Here we repeat both tests without the 200-µV artifact rule, keeping every window.

In [ ]:
everything = np.ones(len(S), dtype=bool)
for name, people in [("experts", experts), ("non-experts", novices)]:
    for band_name, (band, region, where) in REGIONS.items():
        for rule, use in [("200 µV rule", keep), ("no rule    ", everything)]:
            med, mw, _ = band_power_db(people, band, use)
            _, p_values = cluster_test(med - mw)
            d = region_difference(people, band, region, use)
            print(f"{name:11s} {band_name}, {rule}: smallest cluster p = {min(p_values):.2f}, "
                  f"region {d.mean():+.2f} dB, p = {stats.ttest_1samp(d, 0).pvalue:.2f}")

Without the artifact rule, the two borderline results get weaker: the experts' theta region goes from p = 0.04 to p = 0.10, and the non-experts' alpha cluster from p = 0.01 to p = 0.05. For the experts, the direction stays the same in both bands and both versions. **Summary:** the experts show the paper's pattern in direction, location and size, but in our simpler analysis (Welch spectra, an automatic artifact rule, no ICA) it is not clearly above noise; the paper's p < 0.02 we do not reproduce.

## Bonus: predict the answer from 10 s of EEG

The head maps compare averages over many windows. A harder question: from a single 10-s window, can a classifier tell whether the person will report meditation or mind wandering?

- **Features:** two numbers per window, taken from the paper's finding: theta power (dB) in the mid-frontal region and alpha power (dB) in the right central region.
- **Each person is their own reference:** we subtract each person's mean and divide by their spread (a z-score), so the classifier sees "more alpha than usual for this person", not "a person with strong alpha". This uses no answers, only the EEG.
- **Classifier:** logistic regression, $P(\text{meditation}) = 1 / (1 + e^{-(w_1 z_\theta + w_2 z_\alpha + b)})$.
- **Test:** leave one person out. Train on all other people of the group, predict every window of the left-out person, repeat for each person. So the classifier always meets a person it has never seen.
- **Score:** balanced accuracy, the mean of the hit rates for meditation and for mind wandering. Guessing gives 50 %, even though meditation answers are more common.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

def region_db(band, region):
    """Band power (dB) of every window, averaged over the region's channels."""
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    idx = [CH_NAMES.index(ch) for ch in region]
    return (10 * np.log10(S[:, :, in_band].mean(axis=2)))[:, idx].mean(axis=1)

use = keep & (labels_all != "equal") & ~np.isin(person, ["sub-017", "sub-018"])
X = np.column_stack([region_db(THETA, REGIONS["theta"][1]), region_db(ALPHA, REGIONS["alpha"][1])])[use]
y = (labels_all[use] == "med").astype(int)          # 1 = meditation, 0 = mind wandering
who = person[use]

Z = np.empty_like(X)
for sub in np.unique(who):
    mine = who == sub
    Z[mine] = (X[mine] - X[mine].mean(axis=0)) / X[mine].std(axis=0)
print(f"{len(y)} windows, {y.sum()} meditation, {len(y) - y.sum()} mind wandering")

Leave one person out, for experts and for non-experts separately (their alpha differences point in opposite directions). To know what chance looks like here, we repeat everything 200 times with the answers shuffled within each person: same number of meditation and mind-wandering answers, but no link to the EEG.

In [ ]:
def leave_one_person_out(Z, y, people):
    """Balanced accuracy when each person is predicted by a model trained on the others."""
    predicted, truth = [], []
    for sub in people:
        train = np.isin(who, people) & (who != sub)
        test = who == sub
        model = LogisticRegression(class_weight="balanced").fit(Z[train], y[train])
        predicted.append(model.predict(Z[test]))
        truth.append(y[test])
    return balanced_accuracy_score(np.concatenate(truth), np.concatenate(predicted))

def shuffled(y):
    """The answers shuffled within each person."""
    y_shuffled = y.copy()
    for sub in np.unique(who):
        mine = who == sub
        y_shuffled[mine] = rng_shuffle.permutation(y[mine])
    return y_shuffled

rng_shuffle = np.random.default_rng(1)
scores = {}
for name, group_people in [("experts", experts), ("non-experts", novices)]:
    people = [s for s in group_people if s in who]
    real = leave_one_person_out(Z, y, people)
    chance = np.array([leave_one_person_out(Z, shuffled(y), people) for _ in range(200)])
    scores[name] = (real, chance)
    print(f"{name}: balanced accuracy {100 * real:.1f} %, shuffled answers {100 * chance.mean():.1f} % "
          f"(95 % below {100 * np.percentile(chance, 95):.1f} %), p = {np.mean(chance >= real):.3f}")

Left: every expert window as a point in the two features, with the line where a model trained on all experts switches its prediction. Right: the real balanced accuracy against the 200 shuffled versions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1, 1.2]})

ax = axes[0]
is_expert = np.isin(who, experts)
for value, color, text in [(1, BLUE, "meditation"), (0, ORANGE, "mind wandering")]:
    m = is_expert & (y == value)
    ax.scatter(Z[m, 0], Z[m, 1], s=14, color=color, alpha=0.6, lw=0, label=f"{text} ({m.sum()})")
model = LogisticRegression(class_weight="balanced").fit(Z[is_expert], y[is_expert])
(w1, w2), b = model.coef_[0], model.intercept_[0]
zt = np.linspace(-3, 3, 2)
ax.plot(zt, -(w1 * zt + b) / w2, color=GRAY, ls="--", lw=1.5, label="where the model switches")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_xlabel("theta, mid-frontal (z, per person)")
ax.set_ylabel("alpha, right central (z, per person)")
ax.set_title("Experts: one point per 10-s window", fontsize=11)
ax.legend(frameon=False, loc="upper left", fontsize=9)

ax = axes[1]
for row, (name, (real, chance)) in enumerate(scores.items()):
    ax.scatter(100 * chance, row + rng_shuffle.uniform(-0.15, 0.15, len(chance)), s=8, color=GRAY, alpha=0.4, lw=0)
    ax.plot([100 * real] * 2, [row - 0.3, row + 0.3], color="black", lw=3)
    ax.text(100 * real + 0.6, row + 0.3, f"{100 * real:.0f} %, p = {np.mean(chance >= real):.2f}",
            va="center", fontsize=9)
ax.axvline(50, color=GRAY, lw=1, ls=":")
ax.set_yticks(range(len(scores)), list(scores))
ax.set_ylim(-0.6, len(scores) - 0.4)
ax.invert_yaxis()                     # experts on top
ax.set_xlabel("balanced accuracy, left-out people (%)")
ax.set_title("Real answers (black) vs. 200 shuffles (gray)", fontsize=11)

fig.tight_layout()
fig.savefig(FIG_DIR / "classifier.png", dpi=150)
plt.show()

From a single window, the experts' answers cannot be predicted: 53 % balanced accuracy, inside the range of the shuffled answers (p = 0.26). The scatter plot shows why: meditation and mind-wandering windows overlap almost completely. The group difference of about 0.5 dB is small next to the spread of single windows within one person (1 to 2 dB), so it only shows up in averages over many windows. For the non-experts, the classifier reaches 59 % (p = 0.01): weak, but above chance, in line with their alpha difference in the head maps. Their mind-wandering windows also contained far more artifacts, so part of this may come from movement rather than from the mental state.

## Two observations from the answers

Two checks that use only the answers, no EEG. Both matter for the EEG comparison: if tiredness or the time in the session differ between meditation and mind-wandering windows, the two piles differ in more than the mental state.

**Tiredness.** Q3 asked how tired the person is (0 to 3). Is a person more tired when they report mind wandering than when they report meditation? As in the EEG analysis, each person is compared with themself.

In [ ]:
rows = []   # one row per meditation or mind-wandering answer: group, person, minutes into the recording, label, tiredness
for sub in group:
    for ses in sessions[sub]:
        answers_ses, _, _ = find_answers(sub, ses)
        for p in answers_ses:
            if all(q in p for q in ("q1", "q2", "q3")) and label(p["q1"], p["q2"]):
                rows.append((group[sub], sub, p["t"] / 60, label(p["q1"], p["q2"]), p["q3"]))

for name, people in [("experts", experts), ("non-experts", novices)]:
    diffs = []
    for sub in people:
        med = [r[4] for r in rows if r[1] == sub and r[3] == "med"]
        mw = [r[4] for r in rows if r[1] == sub and r[3] == "mw"]
        if med and mw:                                  # people with both kinds of answer
            diffs.append(np.mean(mw) - np.mean(med))
    diffs = np.array(diffs)
    _, p = stats.ttest_1samp(diffs, 0)
    print(f"{name}: more tired before mind wandering in {np.sum(diffs > 0)} of {len(diffs)} people, "
          f"mean difference {diffs.mean():+.2f} points (scale 0-3), p = {p:.2f}")

Most people are more tired before they report mind wandering: 9 of 12 experts (+0.31 points on the 0–3 scale, p = 0.02) and 8 of 10 non-experts (+0.43, p = 0.01). The paper also reports that mind wandering and drowsiness go together. Since tiredness changes theta and alpha by itself, part of the EEG difference may come from tiredness. A test would be to repeat the comparison with only the windows in which the person reported no tiredness (Q3 = 0).

**Time in the session.** Does the share of meditation answers change over a recording? Here pooled over all people of a group, in steps of 10 minutes.

In [ ]:
edges = [0, 10, 20, 30, 40, np.inf]   # minutes into the recording
for name, g in [("experts", "expert"), ("non-experts", "novice")]:
    parts = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        here = np.array([r[3] for r in rows if r[0] == g and lo <= r[2] < hi])
        span = f"{lo}-{hi} min" if hi < np.inf else f"after {lo} min"
        parts.append(f"{span}: {100 * np.mean(here == 'med'):.0f} % ({len(here)})")
    print(f"{name}: " + ", ".join(parts))

The experts' share of meditation rises over the first half hour (73 % to 85 %) and falls after 40 minutes (66 %); the non-experts stay near 50 % and drop to 19 % after 40 minutes. The paper also finds that the experts' meditation depth rises over the first 20 questions. Two cautions: the answers are pooled over people, and only the longer recordings reach the late times.